In [0]:
spark.conf.set(
  
)

In [0]:
# STEP 1 – Imports + Load Gold Data

from pyspark.sql.functions import col, when
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

# 1) Load your Gold features_v1 Delta table
gold_df = spark.read.format("delta").load(
    "abfss://lakehouse@airlinedata60104384.dfs.core.windows.net/gold/flights/features_v1/"
)

print("Gold rows:", gold_df.count())
gold_df.printSchema()
display(gold_df.limit(5))


Gold rows: 2945859
root
 |-- Year: integer (nullable = true)
 |-- Month: integer (nullable = true)
 |-- DayofMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- DepDelay: double (nullable = true)
 |-- ArrDelay: double (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- Distance: double (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- DelayCategory: string (nullable = true)
 |-- AvgDepDelayCarrier: double (nullable = true)
 |-- AvgDepDelayOrigin: double (nullable = true)



Year,Month,DayofMonth,DayOfWeek,FL_DATE,DepDelay,ArrDelay,DepHour,Distance,Origin,Dest,UniqueCarrier,DelayCategory,AvgDepDelayCarrier,AvgDepDelayOrigin
1999,3,1,1,1999-03-01,-2.0,7.0,7,1310.0,MCO,MSP,NW,OnTime,6.47,8.01
1996,5,12,7,1996-05-12,-1.0,6.0,17,954.0,IAD,MSY,UA,OnTime,10.62,10.3
2000,11,7,2,2000-11-07,-3.0,33.0,14,224.0,DFW,IAH,AA,OnTime,8.57,9.5
1999,7,15,4,1999-07-15,-1.0,-2.0,19,455.0,DEN,BIL,UA,OnTime,10.62,9.25
2004,2,23,1,2004-02-23,-4.0,-9.0,10,761.0,LGA,ATL,DL,OnTime,7.66,8.97


In [0]:
# STEP 2 – Create binary label: 0 = OnTime, 1 = Delayed (Minor or Major)

binary_df = (
    gold_df
    .withColumn(
        "label",
        when(col("DelayCategory") == "OnTime", 0).otherwise(1)
    )
)

# Sanity check: labels must be only 0 and 1
binary_df.groupBy("DelayCategory", "label").count().show()
binary_df.groupBy("label").count().show()


+-------------+-----+-------+
|DelayCategory|label|  count|
+-------------+-----+-------+
|       OnTime|    0|1767353|
|        Minor|    1| 689175|
|        Major|    1| 489331|
+-------------+-----+-------+

+-----+-------+
|label|  count|
+-----+-------+
|    1|1178506|
|    0|1767353|
+-----+-------+



In [0]:
# STEP 3 – Select features we will use

numeric_features = [
    "Month",
    "DayOfMonth",
    "DayOfWeek",
    "DepHour",
    "Distance",
    "AvgDepDelayCarrier",
    "AvgDepDelayOrigin"
]

categorical_features = [
    "Origin",
    "Dest",
    "UniqueCarrier"
]

all_needed_cols = numeric_features + categorical_features + ["label"]

final_df = binary_df.select(*all_needed_cols).na.drop()

print("Final dataset rows:", final_df.count())
final_df.printSchema()
display(final_df.limit(5))


Final dataset rows: 2945859
root
 |-- Month: integer (nullable = true)
 |-- DayOfMonth: integer (nullable = true)
 |-- DayOfWeek: integer (nullable = true)
 |-- DepHour: integer (nullable = true)
 |-- Distance: double (nullable = true)
 |-- AvgDepDelayCarrier: double (nullable = true)
 |-- AvgDepDelayOrigin: double (nullable = true)
 |-- Origin: string (nullable = true)
 |-- Dest: string (nullable = true)
 |-- UniqueCarrier: string (nullable = true)
 |-- label: integer (nullable = false)



Month,DayOfMonth,DayOfWeek,DepHour,Distance,AvgDepDelayCarrier,AvgDepDelayOrigin,Origin,Dest,UniqueCarrier,label
10,17,6,9,762.0,8.57,9.5,DFW,IND,AA,1
1,30,2,18,1491.0,10.62,9.25,DEN,BWI,UA,0
6,4,5,17,1068.0,8.57,7.65,PIT,DFW,AA,0
2,19,1,16,272.0,7.66,7.24,PNS,ATL,DL,0
6,15,7,14,584.0,9.68,13.05,ORD,BHM,DH,0


In [0]:
# STEP 4 – Train/Test Split (70% train, 30% test)

train_df, test_df = final_df.randomSplit([0.7, 0.3], seed=42)

print("Train rows:", train_df.count())
print("Test rows :", test_df.count())

train_df.groupBy("label").count().show()
test_df.groupBy("label").count().show()


Train rows: 2062602
Test rows : 883257
+-----+-------+
|label|  count|
+-----+-------+
|    1| 825150|
|    0|1237452|
+-----+-------+

+-----+------+
|label| count|
+-----+------+
|    1|353356|
|    0|529901|
+-----+------+



In [0]:
# STEP 5 – Helper to build a full pipeline with any classifier

def build_pipeline(classifier):
    # Index categorical columns
    indexers = [
        StringIndexer(
            inputCol=c,
            outputCol=f"{c}_idx",
            handleInvalid="keep"
        )
        for c in categorical_features
    ]
    
    # One-hot encode indexed categorical columns
    encoders = [
        OneHotEncoder(
            inputCol=f"{c}_idx",
            outputCol=f"{c}_ohe"
        )
        for c in categorical_features
    ]
    
    # Features = encoded categoricals + numeric features
    assembler_inputs = [f"{c}_ohe" for c in categorical_features] + numeric_features
    
    assembler = VectorAssembler(
        inputCols=assembler_inputs,
        outputCol="features"
    )
    
    stages = indexers + encoders + [assembler, classifier]
    return Pipeline(stages=stages)


In [0]:
# STEP 6 – Define models

# 1) Logistic Regression (baseline)
lr = LogisticRegression(
    labelCol="label",
    featuresCol="features",
    maxIter=20,
    regParam=0.1,
    elasticNetParam=0.0  # pure L2
)

# 2) Random Forest (strong non-linear model)
rf = RandomForestClassifier(
    labelCol="label",
    featuresCol="features",
    numTrees=100,
    maxDepth=10,
    seed=42
)

# Build pipelines
lr_pipeline = build_pipeline(lr)
rf_pipeline = build_pipeline(rf)


In [0]:
# STEP 7 – Train both models

# Logistic Regression
lr_model = lr_pipeline.fit(train_df)
lr_predictions = lr_model.transform(test_df)

# Random Forest
rf_model = rf_pipeline.fit(train_df)
rf_predictions = rf_model.transform(test_df)

print("LR prediction sample:")
display(lr_predictions.select("label", "prediction", "probability").limit(5))

print("RF prediction sample:")
display(rf_predictions.select("label", "prediction", "probability").limit(5))


🏃 View run traveling-panda-757 at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976/runs/f6eddbc617b243e0897965ce8588ea04
🧪 View experiment at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976
🏃 View run crawling-robin-317 at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976/runs/085e889348204003899bd68a91361996
🧪 View experiment at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976
LR prediction sample:


label,prediction,probability
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6456135030764866, 0.3543864969235134))"
0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.7319790106061314, 0.26802098939386865))"
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.633778767110128, 0.36622123288987196))"
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6893736432063562, 0.3106263567936438))"
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.7214617482028145, 0.2785382517971855))"


RF prediction sample:


label,prediction,probability
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6201019191459476, 0.3798980808540524))"
0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6543867918461481, 0.3456132081538518))"
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.60359163047591, 0.39640836952409))"
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6380054195499733, 0.36199458045002664))"
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6748395379473839, 0.32516046205261606))"


In [0]:
# STEP 8 – Evaluation: Accuracy, F1, AUC

acc_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

auc_eval = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

def evaluate_model(name, preds_df):
    acc = acc_eval.evaluate(preds_df)
    f1 = f1_eval.evaluate(preds_df)
    auc = auc_eval.evaluate(preds_df)
    
    print(f"\n===== {name} =====")
    print(f"Accuracy : {acc:.4f}")
    print(f"F1-score : {f1:.4f}")
    print(f"AUC ROC  : {auc:.4f}")
    
    print("Confusion matrix (label vs prediction):")
    preds_df.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# Evaluate both
evaluate_model("Logistic Regression", lr_predictions)
evaluate_model("Random Forest", rf_predictions)



===== Logistic Regression =====
Accuracy : 0.6177
F1-score : 0.5536
AUC ROC  : 0.6336
Confusion matrix (label vs prediction):
+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|    0|       0.0|482220|
|    0|       1.0| 47681|
|    1|       0.0|289949|
|    1|       1.0| 63407|
+-----+----------+------+


===== Random Forest =====
Accuracy : 0.6014
F1-score : 0.4549
AUC ROC  : 0.6382
Confusion matrix (label vs prediction):
+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|    0|       0.0|529064|
|    0|       1.0|   837|
|    1|       0.0|351228|
|    1|       1.0|  2128|
+-----+----------+------+



In [0]:
# ======================================
# NEW MODELS: GBTClassifier + LinearSVC
# ======================================

from pyspark.ml.classification import GBTClassifier, LinearSVC

# 1) Gradient Boosted Trees
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxIter=50,
    maxDepth=8,
    subsamplingRate=0.8,
    stepSize=0.1,
    seed=42
)
gbt_pipeline = build_pipeline(gbt)

# 2) Linear SVM
svm = LinearSVC(
    labelCol="label",
    featuresCol="features",
    maxIter=20,
    regParam=0.1
)
svm_pipeline = build_pipeline(svm)


In [0]:
# Train GBT
gbt_model = gbt_pipeline.fit(train_df)
gbt_predictions = gbt_model.transform(test_df)

# Train SVM
svm_model = svm_pipeline.fit(train_df)
svm_predictions = svm_model.transform(test_df)

print("GBT sample predictions:")
display(gbt_predictions.select("label", "prediction", "probability").limit(5))

print("SVM sample predictions:")
display(svm_predictions.select("label", "prediction").limit(5))


🏃 View run unique-rook-96 at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976/runs/83236b64290b4d96b877174f5b793bfb
🧪 View experiment at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976
🏃 View run redolent-bass-746 at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976/runs/4e072dc845d8415daabffba9465e2dad
🧪 View experiment at: https://adb-4211121333195033.13.azuredatabricks.net/ml/experiments/1274874260597976
GBT sample predictions:


label,prediction,probability
1,1.0,"Map(vectorType -> dense, length -> 2, values -> List(0.4987134000765362, 0.5012865999234638))"
0,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.6482240692651152, 0.3517759307348848))"
1,0.0,"Map(vectorType -> dense, length -> 2, values -> List(0.5296439453627565, 0.4703560546372435))"
1,1.0,"Map(vectorType -> dense, length -> 2, values -> List(0.4787061803239947, 0.5212938196760053))"
1,1.0,"Map(vectorType -> dense, length -> 2, values -> List(0.40319479638329464, 0.5968052036167053))"


SVM sample predictions:


label,prediction
1,0.0
0,0.0
1,0.0
1,0.0
1,0.0


In [0]:
evaluate_model("Gradient Boosted Trees", gbt_predictions)
evaluate_model("Linear SVM", svm_predictions)



===== Gradient Boosted Trees =====
Accuracy : 0.6508
F1-score : 0.6302
AUC ROC  : 0.6792
Confusion matrix (label vs prediction):
+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|    0|       0.0|441329|
|    0|       1.0| 88572|
|    1|       0.0|219857|
|    1|       1.0|133499|
+-----+----------+------+


===== Linear SVM =====
Accuracy : 0.6000
F1-score : 0.4507
AUC ROC  : 0.6158
Confusion matrix (label vs prediction):
+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|    0|       0.0|529588|
|    0|       1.0|   313|
|    1|       0.0|352991|
|    1|       1.0|   365|
+-----+----------+------+

